# Transformer Coherence Model Experiments

Transformer model variants trained and evaluated on the same saved train/validation/test splits produced by the classical experiment notebook.


In [2]:
import json
import logging
import random
import time
from pathlib import Path
from typing import Dict, Iterable, Optional

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup


SEED = 42
DEFAULT_MODELS = [
    "distilbert-base-uncased",
    "bert-base-uncased",
    "roberta-base",
    "microsoft/deberta-base",
]

# DeBERTa base stores its default weights as a legacy .bin file. This safetensors
# revision avoids the torch<2.6 loading restriction in recent transformers versions.
MODEL_REVISIONS = {
    "microsoft/deberta-base": "refs/pr/4",
}


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


In [3]:
class StoryDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length: int):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


In [4]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [5]:
def metric_dict(y_true: Iterable[int], y_pred: Iterable[int], y_score: Iterable[float]) -> Dict[str, float]:
    y_true = np.asarray(list(y_true))
    y_pred = np.asarray(list(y_pred))
    y_score = np.asarray(list(y_score))
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score),
    }


In [6]:
def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    total = 0
    for batch in tqdm(loader, desc="train", leave=False):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * labels.size(0)
        total += labels.size(0)
    return total_loss / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    preds, scores, labels = [], [], []
    start = time.time()
    for batch in tqdm(loader, desc="eval", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        batch_labels = batch["labels"].numpy()
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = torch.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()
        batch_preds = logits.argmax(dim=-1).detach().cpu().numpy()
        labels.extend(batch_labels)
        scores.extend(probs)
        preds.extend(batch_preds)
    elapsed = time.time() - start
    return preds, scores, labels, elapsed


In [7]:
def load_split(split_dir: Path, name: str, quick: bool) -> pd.DataFrame:
    df = pd.read_csv(split_dir / f"{name}.csv")
    if quick:
        df = df.sample(n=min(600, len(df)), random_state=SEED).reset_index(drop=True)
    return df


In [8]:
def run_model(
    model_name: str,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    output_dir: Path,
    max_length: int,
    batch_size: int,
    epochs: int,
    learning_rate: float,
    warmup_ratio: float,
    weight_decay: float,
    device,
) -> Dict[str, float]:
    safe_name = model_name.replace("/", "__")
    model_dir = output_dir / safe_name
    model_dir.mkdir(parents=True, exist_ok=True)

    revision = MODEL_REVISIONS.get(model_name)
    load_kwargs = {"revision": revision} if revision else {}
    model_load_kwargs = {**load_kwargs, "use_safetensors": True} if revision else load_kwargs

    log.info("Loading %s%s", model_name, f" @ {revision}" if revision else "")
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, **load_kwargs)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        **model_load_kwargs,
    ).to(device)

    train_loader = DataLoader(
        StoryDataset(train_df["story"], train_df["label"], tokenizer, max_length),
        batch_size=batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        StoryDataset(val_df["story"], val_df["label"], tokenizer, max_length),
        batch_size=batch_size,
        shuffle=False,
    )
    test_loader = DataLoader(
        StoryDataset(test_df["story"], test_df["label"], tokenizer, max_length),
        batch_size=batch_size,
        shuffle=False,
    )

    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * warmup_ratio),
        num_training_steps=total_steps,
    )

    best_val_f1 = -1.0
    history = []
    best_path = model_dir / "best_model.pt"
    train_start = time.time()
    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(model, train_loader, optimizer, scheduler, device)
        val_pred, val_score, val_label, _ = evaluate(model, val_loader, device)
        val_metrics = metric_dict(val_label, val_pred, val_score)
        history.append({"epoch": epoch, "train_loss": loss, **{f"val_{k}": v for k, v in val_metrics.items()}})
        log.info("%s epoch %s | loss=%.4f val_f1=%.4f", model_name, epoch, loss, val_metrics["f1"])
        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            torch.save(model.state_dict(), best_path)

    training_seconds = time.time() - train_start
    pd.DataFrame(history).to_csv(model_dir / "training_history.csv", index=False)

    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
    test_pred, test_score, test_label, inference_seconds = evaluate(model, test_loader, device)
    test_metrics = metric_dict(test_label, test_pred, test_score)

    pd.DataFrame(
        {
            "label": test_label,
            "prediction": test_pred,
            "coherent_score": test_score,
        }
    ).to_csv(model_dir / "test_predictions.csv", index=False)
    pd.DataFrame(confusion_matrix(test_label, test_pred)).to_csv(model_dir / "test_confusion_matrix.csv", index=False)
    with open(model_dir / "test_classification_report.txt", "w", encoding="utf-8") as f:
        f.write(
            classification_report(
                test_label,
                test_pred,
                target_names=["Incoherent", "Coherent"],
                zero_division=0,
            )
        )

    return {
        "model": model_name,
        "best_val_f1": best_val_f1,
        "training_seconds": training_seconds,
        "test_inference_seconds": inference_seconds,
        "test_examples_per_second": len(test_df) / inference_seconds if inference_seconds else np.nan,
        **{f"test_{k}": v for k, v in test_metrics.items()},
    }


## Run Transformer Experiments

Run the setup cell once, then run each model cell independently. If one model fails, fix only that cell and rerun it.


In [9]:
SPLIT_DIR = Path("outputs/experiments/splits")
OUTPUT_DIR = Path("outputs/transformer_experiments")
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
QUICK = True  # Set False for final reported experiments.

set_seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info("Using device: %s", device)

train_df = load_split(SPLIT_DIR, "train", QUICK)
val_df = load_split(SPLIT_DIR, "val", QUICK)
test_df = load_split(SPLIT_DIR, "test", QUICK)

RESULTS_PATH = OUTPUT_DIR / "transformer_model_comparison.csv"
FAILED_PATH = OUTPUT_DIR / "failed_transformer_models.csv"

if RESULTS_PATH.exists():
    RESULT_ROWS = pd.read_csv(RESULTS_PATH).to_dict("records")
    log.info("Loaded %s previous successful transformer results", len(RESULT_ROWS))
else:
    RESULT_ROWS = []

if FAILED_PATH.exists():
    FAILED_ROWS = pd.read_csv(FAILED_PATH).to_dict("records")
    log.info("Loaded %s previous failed transformer records", len(FAILED_ROWS))
else:
    FAILED_ROWS = []


19:04:52 [INFO] Using device: cuda


In [10]:
def run_and_save_transformer(model_name: str):
    try:
        row = run_model(
            model_name,
            train_df,
            val_df,
            test_df,
            OUTPUT_DIR,
            MAX_LENGTH,
            BATCH_SIZE,
            EPOCHS,
            LEARNING_RATE,
            WARMUP_RATIO,
            WEIGHT_DECAY,
            device,
        )
        RESULT_ROWS[:] = [existing for existing in RESULT_ROWS if existing["model"] != model_name]
        RESULT_ROWS.append(row)
        results = pd.DataFrame(RESULT_ROWS).sort_values("best_val_f1", ascending=False)
        results.to_csv(RESULTS_PATH, index=False)
        if FAILED_PATH.exists():
            failed = pd.read_csv(FAILED_PATH)
            failed = failed[failed["model"] != model_name]
            if len(failed):
                failed.to_csv(FAILED_PATH, index=False)
            else:
                FAILED_PATH.unlink()
        return results
    except Exception as exc:
        log.exception("%s failed", model_name)
        FAILED_ROWS.append({"model": model_name, "error": repr(exc)})
        failed = pd.DataFrame(FAILED_ROWS)
        failed.to_csv(FAILED_PATH, index=False)
        return failed


### DistilBERT


In [18]:
run_and_save_transformer("distilbert-base-uncased")


18:33:14 [INFO] Loading distilbert-base-uncased
18:33:14 [INFO] HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
18:33:14 [INFO] HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
18:33:15 [INFO] HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
18:33:15 [INFO] HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
18:33:15 [INFO] HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
18:33:15 [INFO] HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased/tree/main?recursive=true&expand=false "HT

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
18:33:19 [INFO] distilbert-base-uncased epoch 1 | loss=0.6948 val_f1=0.6674
18:33:23 [INFO] distilbert-base-uncased epoch 2 | loss=0.6896 val_f1=0.6642
18:33:26 [INFO] distilbert-base-uncased epoch 3 | loss=0.6661 val_f1=0.6667


,model,best_val_f1,training_seconds,test_inference_seconds,test_examples_per_second,test_accuracy,test_precision,test_recall,test_f1,test_mcc,test_roc_auc,test_pr_auc
0,distilbert-base-uncased,0.667408,9.470417,0.690258,869.240464,0.501667,0.500835,1.0,0.667408,0.040859,0.553411,0.543219


### BERT Base


In [19]:
run_and_save_transformer("bert-base-uncased")


18:33:26 [INFO] Loading bert-base-uncased
18:33:27 [INFO] HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
18:33:27 [INFO] HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
18:33:27 [INFO] HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
18:33:28 [INFO] HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
18:33:28 [INFO] HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
18:33:28 [INFO] HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
18:33:28 [INFO] HTTP Requ

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
18:33:35 [INFO] bert-base-uncased epo

,model,best_val_f1,training_seconds,test_inference_seconds,test_examples_per_second,test_accuracy,test_precision,test_recall,test_f1,test_mcc,test_roc_auc,test_pr_auc
1,bert-base-uncased,0.700617,17.789615,1.232458,486.831866,0.688333,0.676012,0.723333,0.698873,0.377593,0.753056,0.728197
0,distilbert-base-uncased,0.667408,9.470417,0.690258,869.240464,0.501667,0.500835,1.000000,0.667408,0.040859,0.553411,0.543219


### RoBERTa Base


In [20]:
run_and_save_transformer("roberta-base")


18:33:48 [INFO] Loading roberta-base
18:33:49 [INFO] HTTP Request: HEAD https://huggingface.co/roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
18:33:49 [INFO] HTTP Request: HEAD https://huggingface.co/roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
18:33:49 [INFO] HTTP Request: GET https://huggingface.co/api/models/roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
18:33:49 [INFO] HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
18:33:50 [INFO] HTTP Request: GET https://huggingface.co/api/models/roberta-base/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
18:33:50 [INFO] HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
18:33:50 [INFO] HTTP Request: HEAD https://huggingface.co/robe

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
18:33:57 [INFO] roberta-base epoch 1 | loss=0.6960 val_f1=0.6734
18:34:03 [INFO] roberta-base epoch 2 | loss=0.6

,model,best_val_f1,training_seconds,test_inference_seconds,test_examples_per_second,test_accuracy,test_precision,test_recall,test_f1,test_mcc,test_roc_auc,test_pr_auc
2,roberta-base,0.868098,19.353006,1.163695,515.598976,0.876667,0.834320,0.940000,0.884013,0.759450,0.940000,0.922402
1,bert-base-uncased,0.700617,17.789615,1.232458,486.831866,0.688333,0.676012,0.723333,0.698873,0.377593,0.753056,0.728197
0,distilbert-base-uncased,0.667408,9.470417,0.690258,869.240464,0.501667,0.500835,1.000000,0.667408,0.040859,0.553411,0.543219


### DeBERTa Base

Using `microsoft/deberta-base` avoids the extra SentencePiece dependency required by `microsoft/deberta-v3-base`.


In [11]:
run_and_save_transformer("microsoft/deberta-base")


19:05:07 [INFO] Loading microsoft/deberta-base @ refs/pr/4
19:05:07 [INFO] HTTP Request: HEAD https://huggingface.co/microsoft/deberta-base/resolve/refs%2Fpr%2F4/config.json "HTTP/1.1 307 Temporary Redirect"
19:05:07 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
19:05:08 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/deberta-base/315ff8e6747aba12a2074c165fcc7e47d2cec9b4/config.json "HTTP/1.1 200 OK"
19:05:08 [INFO] HTTP Request: HEAD https://huggingface.co/microsoft/deberta-base/resolve/refs%2Fpr%2F4/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
19:05:08 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/deberta-base/315ff8e6747aba12a2074c165fcc7e47d2cec9b4/tokenizer_config.json "HTTP/1.1 200 OK"
19:05:08 [INFO] HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-base/tree/refs%2Fpr%2F4/ad

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
19:05:21 [INFO] microsoft/deberta-base epoch 1 | loss=0.6541 v

,model,best_val_f1,training_seconds,test_inference_seconds,test_examples_per_second,test_accuracy,test_precision,test_recall,test_f1,test_mcc,test_roc_auc,test_pr_auc
0,microsoft/deberta-base,0.86747,24.579355,1.644937,364.755602,0.846667,0.787293,0.95,0.861027,0.708632,0.920256,0.896616


### Results Summary


In [14]:
if RESULTS_PATH.exists():
    final_transformer_results = pd.read_csv(RESULTS_PATH).sort_values("best_val_f1", ascending=False)
    display(final_transformer_results)
else:
    print("No successful transformer results saved yet.")

if FAILED_PATH.exists():
    print("Failed models:")
    display(pd.read_csv(FAILED_PATH))


,model,best_val_f1,training_seconds,test_inference_seconds,test_examples_per_second,test_accuracy,test_precision,test_recall,test_f1,test_mcc,test_roc_auc,test_pr_auc
0,roberta-base,0.868098,19.099746,1.086863,552.047356,0.876667,0.834320,0.940000,0.884013,0.759450,0.940000,0.922402
1,microsoft/deberta-base,0.867470,24.579355,1.644937,364.755602,0.846667,0.787293,0.950000,0.861027,0.708632,0.920256,0.896616
2,bert-base-uncased,0.700617,17.294973,1.224449,490.016453,0.688333,0.676012,0.723333,0.698873,0.377593,0.753056,0.728197
3,distilbert-base-uncased,0.667408,9.647137,0.638295,940.004677,0.501667,0.500835,1.000000,0.667408,0.040859,0.553411,0.543219


In [ ]:
with open(OUTPUT_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "split_dir": str(SPLIT_DIR),
            "output_dir": str(OUTPUT_DIR),
            "max_length": MAX_LENGTH,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "warmup_ratio": WARMUP_RATIO,
            "weight_decay": WEIGHT_DECAY,
            "quick": QUICK,
        },
        f,
        indent=2,
    )


: 